# 08 — MQTT and in-process depth scaling

MQTT-bookended and in-process populations remain separate. The comparison is descriptive and does not subtract unmatched runs. N is independent runs, units are microseconds, and missing depths remain PENDING, never zero.


In [ ]:
import os
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
from wafer_analysis.focused import evidence_label, passed_artifacts, pending_record, percentile_rows
from wafer_analysis.paths import resolve_analysis_batch

def resolve(experiment, env_name):
    return resolve_analysis_batch(experiment, os.environ.get(env_name))[0]

rows=[]
for experiment,kind,env in [('e-perf-3','MQTT bookends','E_PERF_3_DIR'),('e-perf-8','in-process','E_PERF_8_DIR')]:
    batch=resolve(experiment,env); df=pd.DataFrame() if batch is None else percentile_rows(batch)
    for depth in ['depth-1','depth-3','depth-5','depth-10']:
        values=df[df.condition==depth] if not df.empty else pd.DataFrame()
        if values.empty:
            row=pending_record(f'{kind} {depth}','no passed percentile leaf','microseconds'); row.update({'kind':kind,'condition':depth}); rows.append(row)
        else: rows.append({'question':f'{kind} {depth}','kind':kind,'condition':depth,'status':'READY','N':len(values),'value':values.p50_ns.median()/1e3,'units':'p50 microseconds','uncertainty':'descriptive only','thesis_evidence':False})
out=pd.DataFrame(rows); display(out)
ready=out[out.status=='READY']
if not ready.empty:
    for kind,group in ready.groupby('kind'): plt.plot(group.condition,group.value,marker='o',label=kind)
    plt.ylabel('Median run p50 (µs)'); plt.title('Depth scaling — diagnostic'); plt.legend()
